# 02. 데이터 수집

**목적**: `01_universe_selection.ipynb` 에서 확정한 110개 종목에 대해 다음 데이터를 수집한다.

1. **개별 종목 가격** — yfinance `Ticker.history()` (2010-01-01 ~ 2025-12-31)
2. **Fama-French 5 Factor + MOM** — Ken French 사이트 ZIP 직접 다운로드
3. **벤치마크** — `^GSPC`, `SPY`, SPDR 섹터 ETF 11종

**산출물**
- `data/cache/{ticker}.pkl` — 개별 종목 가격 원본 (110개)
- `data/ff_factors.csv` — 일간 FF5 + MOM 팩터 (7개 컬럼)
- `data/benchmarks/{symbol}.csv` — 벤치마크 가격 (13개)

> **주의**: 가격 수집은 yfinance API 110회 호출 → 네트워크 상태에 따라 수 분 소요. 중간에 중단해도 캐시(.pkl) 덕분에 재실행 시 이어서 진행된다.

---
## Section 0. 설정 및 함수 정의

In [ ]:
# 표준 라이브러리 + 외부 패키지 (pandas, numpy, yfinance, requests)
import io
import os
import pickle
import platform
import re
import time
import zipfile
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import yfinance as yf

# ── 경로 설정 ────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR     = PROJECT_DIR / 'data'
CACHE_DIR    = DATA_DIR / 'cache'
BENCH_DIR    = DATA_DIR / 'benchmarks'

for d in [DATA_DIR, CACHE_DIR, BENCH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── 수집 기간 (사용자 확정: 2010-01-01 ~ 2025-12-31, 16년) ──────────
PRICE_START = '2010-01-01'
PRICE_END   = '2025-12-31'


def setup_korean_font():
    """matplotlib 한글 폰트 설정 — OS별 분기."""
    if platform.system() == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    elif platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    else:
        try:
            import koreanize_matplotlib  # noqa: F401
        except ImportError:
            pass
    plt.rcParams['axes.unicode_minus'] = False


setup_korean_font()


def safe_download_price(
    ticker: str,
    start: str = PRICE_START,
    end: str = PRICE_END,
    max_retry: int = 5,
) -> Optional[pd.DataFrame]:
    """yfinance 개별 Ticker.history 호출 + 지수 백오프 재시도.

    yfinance 의 rate-limit(429) 이나 일시적 네트워크 오류에 대비해
    최대 max_retry 회 재시도한다 (대기: 1 → 2 → 4 → 8 → 16s).

    반환: Close/High/Low/Open/Volume/Dividends/Stock Splits DataFrame
          타임존 제거, index.name='date'
          실패 시 None
    """
    for attempt in range(max_retry + 1):
        try:
            tk = yf.Ticker(ticker)
            df = tk.history(
                start=start,
                end=end,
                auto_adjust=False,
                actions=True,  # Dividends, Stock Splits 포함
            )
            if df.empty:
                return None
            # 타임존 제거 — timezone-naive 날짜 인덱스로 통일
            if df.index.tz is not None:
                df.index = df.index.tz_localize(None)
            df.index.name = 'date'
            return df
        except Exception:
            if attempt < max_retry:
                time.sleep(2 ** attempt)  # 지수 백오프
    return None


def download_ff_zip(url: str) -> pd.DataFrame:
    """Ken French 사이트에서 ZIP을 받아 일간 팩터 DataFrame 반환.

    처리:
      1) ZIP 안 첫 번째 CSV 추출
      2) YYYYMMDD 8자리로 시작하는 행을 찾아 헤더·데이터 블록 파싱
      3) 날짜: YYYYMMDD 정수 → datetime 인덱스
      4) 값: 퍼센트(%) → 소수 (/100)
      5) 연간 집계 푸터 자동 제거
    """
    r = requests.get(url, timeout=60)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        csv_name = zf.namelist()[0]
        raw = zf.read(csv_name).decode('utf-8', errors='ignore')

    lines = raw.splitlines()

    # YYYYMMDD 형식으로 시작하는 첫 번째 데이터 행 위치
    start_idx = next(
        i for i, ln in enumerate(lines)
        if re.match(r'^\s*\d{8}\s*,', ln)
    )
    # 데이터 블록 끝 (YYYYMMDD 패턴이 끊기는 지점)
    end_idx = next(
        (i for i in range(start_idx, len(lines))
         if not re.match(r'^\s*\d{8}\s*,', lines[i])),
        len(lines)
    )
    # start_idx - 1: 바로 위 행이 컬럼 헤더
    data_block = '\n'.join(lines[start_idx - 1 : end_idx])

    df = pd.read_csv(io.StringIO(data_block))
    df.columns = [c.strip() for c in df.columns]
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(
        df[date_col].astype(int).astype(str), format='%Y%m%d'
    )
    df = df.rename(columns={date_col: 'date'}).set_index('date')
    return df.astype(float) / 100.0  # % → 소수


def download_benchmark(
    symbol: str,
    start: str = PRICE_START,
    end: str = PRICE_END,
) -> Optional[pd.DataFrame]:
    """yfinance 로 벤치마크(지수/ETF) 일간 가격 수집.

    safe_download_price 를 재사용하되 재시도 횟수를 줄임(3회).
    """
    return safe_download_price(symbol, start, end, max_retry=3)


print(f'프로젝트 경로 : {PROJECT_DIR}')
print(f'수집 기간     : {PRICE_START} ~ {PRICE_END}')
print(f'캐시 경로     : {CACHE_DIR}')
print(f'벤치마크 경로 : {BENCH_DIR}')

---
## Section 1. 개별 종목 가격 수집 (yfinance)

**수집 방식**
- `Ticker.history(auto_adjust=False, actions=True)`: 조정 전 OHLCV + 배당·분할 정보 동시 수집
- 이미 캐시(.pkl)가 있는 종목은 스킵 — 중단 후 재실행해도 이어서 진행
- Rate-limit 방지: 종목 간 0.3s 간격 + 지수 백오프 재시도

> **예상 소요 시간**: 캐시 없이 110종목 처음 실행 시 약 5~15분 (네트워크 상태에 따라 변동)

In [ ]:
# universe.csv 로드
universe_path = DATA_DIR / 'universe.csv'
assert universe_path.exists(), (
    f'universe.csv 없음: {universe_path}\n'
    '→ 01_universe_selection.ipynb 를 먼저 실행하세요'
)

df_universe = pd.read_csv(universe_path)
print(f'유니버스 종목 수: {len(df_universe)}')
print('\n섹터별 종목 수:')
print(df_universe['gics_sector'].value_counts().to_string())

tickers = df_universe['ticker'].tolist()

# 가격 다운로드 루프
log_ok   = []
log_fail = []

for i, ticker in enumerate(tickers, 1):
    cache_path = CACHE_DIR / f'{ticker}.pkl'

    # 이미 캐시 있으면 스킵
    if cache_path.exists():
        log_ok.append(ticker)
        if i % 30 == 0 or i == len(tickers):
            print(f'  [{i:3d}/{len(tickers)}] (캐시 스킵 중... {ticker})')
        continue

    # 다운로드
    df_price = safe_download_price(ticker)

    if df_price is None or df_price.empty:
        log_fail.append(ticker)
        print(f'  [{i:3d}/{len(tickers)}] {ticker:8s} → FAIL')
    else:
        with open(cache_path, 'wb') as f:
            pickle.dump(df_price, f)
        log_ok.append(ticker)
        if i % 10 == 0 or i == len(tickers):
            print(f'  [{i:3d}/{len(tickers)}] {ticker:8s} → OK ({len(df_price)}행)')

    time.sleep(0.3)  # API 과부하 방지

print(f'\n완료: 성공 {len(log_ok)}개 / 실패 {len(log_fail)}개')
if log_fail:
    print(f'실패 종목: {log_fail}')

In [ ]:
# 수집 결과 요약 — 날짜 범위 + NaN 비율
summary_records = []
for ticker in tickers:
    cache_path = CACHE_DIR / f'{ticker}.pkl'
    if not cache_path.exists():
        summary_records.append({
            'ticker': ticker, 'rows': 0,
            'start': None, 'end': None, 'nan_close_pct': None, 'status': 'MISSING',
        })
        continue
    with open(cache_path, 'rb') as f:
        df_p = pickle.load(f)
    # 컬럼명 정규화 — yfinance 버전에 따라 'Close'/'close' 혼재 가능
    col = next((c for c in df_p.columns if c.lower() == 'close'), df_p.columns[0])
    summary_records.append({
        'ticker': ticker,
        'rows': len(df_p),
        'start': str(df_p.index[0].date()),
        'end':   str(df_p.index[-1].date()),
        'nan_close_pct': round(df_p[col].isna().mean() * 100, 2),
        'status': 'OK',
    })

df_summary = pd.DataFrame(summary_records)
ok_mask = df_summary['status'] == 'OK'

print('=== 가격 수집 요약 ===')
print(f"성공: {ok_mask.sum()}개")
print(f"실패: {(~ok_mask).sum()}개")
print('\n행 수 분포 (성공 종목):')
print(df_summary.loc[ok_mask, 'rows'].describe().to_string())
print('\n날짜 범위:')
print(f"  가장 이른 start : {df_summary.loc[ok_mask,'start'].min()}")
print(f"  가장 늦은 end   : {df_summary.loc[ok_mask,'end'].max()}")

fail_list = df_summary.loc[~ok_mask, 'ticker'].tolist()
if fail_list:
    print(f'\n실패 티커: {fail_list}')

df_summary.head(20)

---
## Section 2. Fama-French 5 Factor + MOM 수집

**소스**: Ken French 교수 홈페이지 ZIP 직접 다운로드
- FF5 (일간): `F-F_Research_Data_5_Factors_2x3_daily_CSV.zip`
- MOM (일간): `F-F_Momentum_Factor_daily_CSV.zip`

**pandas_datareader 미사용 이유**: 호환 이슈(FRED/French 엔드포인트 불안정). ZIP 직접 다운로드 방식이 더 안정적이다.

**산출 컬럼**: `mkt_rf`, `smb`, `hml`, `rmw`, `cma`, `rf`, `mom_factor` (단위: 소수, 퍼센트가 아님)

In [ ]:
# Ken French 일간 팩터 ZIP URL
FF5_URL = (
    'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'
    'F-F_Research_Data_5_Factors_2x3_daily_CSV.zip'
)
MOM_URL = (
    'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'
    'F-F_Momentum_Factor_daily_CSV.zip'
)

FF_SAVE_PATH = DATA_DIR / 'ff_factors.csv'

print('FF5 팩터 다운로드 중...')
df_ff5 = download_ff_zip(FF5_URL)
# 컬럼명 표준화: 'Mkt-RF' → 'mkt_rf', 'SMB' → 'smb' 등
ff5_rename = {
    'Mkt-RF': 'mkt_rf', 'SMB': 'smb', 'HML': 'hml',
    'RMW': 'rmw', 'CMA': 'cma', 'RF': 'rf',
}
df_ff5.columns = [c.strip() for c in df_ff5.columns]
df_ff5 = df_ff5.rename(columns={k: v for k, v in ff5_rename.items() if k in df_ff5.columns})
print(f'  FF5: {len(df_ff5)}행  컬럼: {df_ff5.columns.tolist()}')

print('MOM 팩터 다운로드 중...')
df_mom = download_ff_zip(MOM_URL)
df_mom.columns = [c.strip() for c in df_mom.columns]
# MOM 컬럼명 표준화 → mom_factor
if 'Mom' in df_mom.columns:
    df_mom = df_mom.rename(columns={'Mom': 'mom_factor'})
elif 'MOM' in df_mom.columns:
    df_mom = df_mom.rename(columns={'MOM': 'mom_factor'})
elif len(df_mom.columns) >= 1:
    df_mom = df_mom.rename(columns={df_mom.columns[0]: 'mom_factor'})
print(f'  MOM: {len(df_mom)}행  컬럼: {df_mom.columns.tolist()}')

# FF5 + MOM 병합 (inner join — 공통 날짜만)
df_ff = df_ff5.join(df_mom[['mom_factor']], how='inner')

# 수집 기간으로 필터
df_ff = df_ff[
    (df_ff.index >= pd.Timestamp(PRICE_START))
    & (df_ff.index <= pd.Timestamp(PRICE_END))
]

# 저장
df_ff.to_csv(FF_SAVE_PATH, encoding='utf-8-sig')
print(f'\n저장: {FF_SAVE_PATH.name}')
print(f'기간: {df_ff.index[0].date()} ~ {df_ff.index[-1].date()}')
print(f'행 수: {len(df_ff)}')
print(f'컬럼: {df_ff.columns.tolist()}')
print(f'\nNaN 수:')
print(df_ff.isna().sum().to_string())
print(f'\n최근 5행:')
df_ff.tail()

---
## Section 3. 벤치마크 수집

| 심볼 | 설명 |
|------|------|
| `^GSPC` | S&P 500 지수 |
| `SPY` | SPDR S&P 500 ETF |
| `XLE` ~ `XLRE` | SPDR 섹터 ETF 11종 (GICS 섹터별 벤치마크) |

이후 `03_feature_build.ipynb` 에서 초과 수익률(Excess Return) 계산 시 사용된다.

In [ ]:
# 벤치마크 목록 (심볼: 설명)
BENCHMARK_SYMBOLS = {
    '^GSPC': 'S&P 500 지수',
    'SPY':   'SPDR S&P 500 ETF',
    'XLE':   'Energy',
    'XLB':   'Materials',
    'XLI':   'Industrials',
    'XLY':   'Consumer Discretionary',
    'XLP':   'Consumer Staples',
    'XLV':   'Health Care',
    'XLF':   'Financials',
    'XLK':   'Information Technology',
    'XLC':   'Communication Services',
    'XLU':   'Utilities',
    'XLRE':  'Real Estate',
}

bench_ok   = []
bench_fail = []

for symbol, desc in BENCHMARK_SYMBOLS.items():
    # ^ 는 파일명에 사용 불가 → 제거 (^GSPC → GSPC.csv)
    save_name = symbol.replace('^', '') + '.csv'
    save_path = BENCH_DIR / save_name

    if save_path.exists():
        bench_ok.append(symbol)
        print(f'  {symbol:6s} ({desc}) → 캐시 스킵')
        continue

    df_bench = download_benchmark(symbol)
    if df_bench is None or df_bench.empty:
        bench_fail.append(symbol)
        print(f'  {symbol:6s} ({desc}) → FAIL')
    else:
        df_bench.to_csv(save_path, encoding='utf-8-sig')
        bench_ok.append(symbol)
        date_range = f'{df_bench.index[0].date()} ~ {df_bench.index[-1].date()}'
        print(f'  {symbol:6s} ({desc}) → OK ({len(df_bench)}행, {date_range})')

    time.sleep(0.5)

print(f'\n벤치마크: 성공 {len(bench_ok)}개 / 실패 {len(bench_fail)}개')
if bench_fail:
    print(f'실패: {bench_fail}')

---
## Section 4. 수집 완료 요약 로그

In [ ]:
print('=' * 60)
print('02_data_collection 수집 완료 요약')
print('=' * 60)

# 가격 캐시 현황
n_price_ok = sum(1 for t in tickers if (CACHE_DIR / f'{t}.pkl').exists())
print(f'\n[1] 가격 캐시  : {n_price_ok} / {len(tickers)}개')
print(f'    경로        : {CACHE_DIR}')

# FF 팩터 현황
if FF_SAVE_PATH.exists():
    df_ff_check = pd.read_csv(FF_SAVE_PATH, index_col='date', nrows=2)
    print(f'\n[2] FF 팩터    : {FF_SAVE_PATH.name} (존재)')
    print(f'    컬럼        : {df_ff_check.columns.tolist()}')
else:
    print('\n[2] FF 팩터    : 파일 없음 — Section 2 셀을 재실행하세요')

# 벤치마크 현황
n_bench = len(list(BENCH_DIR.glob('*.csv')))
print(f'\n[3] 벤치마크   : {n_bench} / {len(BENCHMARK_SYMBOLS)}개 CSV 존재')
print(f'    경로        : {BENCH_DIR}')

print('\n다음 단계: 03_feature_build.ipynb')
print('  → 피처 생성 (수익률/모멘텀/변동성) + 개별 패널 CSV (data/panels/{ticker}.csv)')